# Setup

In [ ]:
"""
Parsing Exploration: SEC 10-K Filing Structure

Goal: Take one full-submission.txt (the gnarly multi-document SGML blob)
and extract the primary 10-K HTML from it.
"""
from pathlib import Path
import re
from collections import Counter

# Notebook lives in notebooks/, so go up one to find project root
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "raw" / "sec-edgar-filings"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir exists: {DATA_DIR.exists()}")
print(f"Tickers found: {sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())}")

# Pick one filing to study

In [ ]:
# We'll start with the most recent AAPL 10-K — the one we already eyeballed yesterday.
ticker = "AAPL"
form = "10-K"

filing_dirs = sorted((DATA_DIR / ticker / form).iterdir())
latest_dir = filing_dirs[-1]
filing_path = latest_dir / "full-submission.txt"

size_mb = filing_path.stat().st_size / (1024 * 1024)
print(f"Filing: {ticker} {form}")
print(f"Accession: {latest_dir.name}")
print(f"Path: {filing_path.relative_to(PROJECT_ROOT)}")
print(f"Size: {size_mb:.2f} MB")

# Load it

In [ ]:
raw = filing_path.read_text(encoding="utf-8", errors="replace")
print(f"Total characters: {len(raw):,}")
print(f"\n--- First 800 chars (the SEC-HEADER block) ---\n")
print(raw[:800])

# Confirm the document count matches the header

In [ ]:
doc_starts = list(re.finditer(r'<DOCUMENT>', raw))
doc_ends = list(re.finditer(r'</DOCUMENT>', raw))

print(f"Opening <DOCUMENT> tags: {len(doc_starts)}")
print(f"Closing </DOCUMENT> tags: {len(doc_ends)}")
# These should match each other AND the PUBLIC DOCUMENT COUNT in the header

# What kinds of documents are inside?

In [ ]:
type_pattern = re.compile(r'<DOCUMENT>\s*<TYPE>([^\n<]+)')
types = [t.strip() for t in type_pattern.findall(raw)]

print(f"Total documents: {len(types)}\n")
print("Top document types in this filing:")
for doc_type, count in Counter(types).most_common(15):
    print(f"  {count:3d}  {doc_type}")

# Extract the primary 10-K document

In [ ]:
# Regex breakdown:
#   <DOCUMENT>          - block opens
#   <TYPE>...           - capture filing type ("10-K")
#   <SEQUENCE>...       - capture sequence number
#   <FILENAME>...       - capture original HTML filename (e.g., aapl-20250927.htm)
#   <DESCRIPTION>...    - optional human description
#   <TEXT>...</TEXT>    - the actual HTML payload (DOTALL so it spans newlines)
#   </DOCUMENT>         - block closes
doc_pattern = re.compile(
    r'<DOCUMENT>\s*'
    r'<TYPE>([^\n<]+)\s*'
    r'<SEQUENCE>([^\n<]+)\s*'
    r'<FILENAME>([^\n<]+)\s*'
    r'(?:<DESCRIPTION>[^\n<]*\s*)?'  # ?: = non-capturing; this group is optional
    r'<TEXT>(.*?)</TEXT>\s*'
    r'</DOCUMENT>',
    re.DOTALL,
)

documents = doc_pattern.findall(raw)
print(f"Successfully parsed {len(documents)} documents (out of {len(types)} expected)")

# Filter to just the primary filing
primary = [d for d in documents if d[0].strip() == form]
print(f"Documents of type '{form}': {len(primary)}")

doc_type, sequence, filename, html_payload = primary[0]
print(f"\n--- Primary 10-K document ---")
print(f"  Type:     {doc_type.strip()}")
print(f"  Sequence: {sequence.strip()}")
print(f"  Filename: {filename.strip()}")
print(f"  HTML size: {len(html_payload):,} chars (~{len(html_payload)/1024:.0f} KB)")

# Save the primary HTML so we can SEE it

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / ticker / form / latest_dir.name
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

primary_html_path = PROCESSED_DIR / "primary.html"
primary_html_path.write_text(html_payload, encoding="utf-8")

print(f"Saved to: {primary_html_path.relative_to(PROJECT_ROOT)}")
print(f"\nOpen it in your browser:")
print(f"  file:///{primary_html_path.as_posix()}")

# Strip HTML to clean text

In [ ]:
from bs4 import BeautifulSoup

soup = BeautifulSoup(html_payload, "lxml")

# Remove iXBRL hidden metadata blocks (they're huge and not user-visible text)
for hidden in soup.find_all(style=lambda s: s and "display:none" in s.lower()):
    hidden.decompose()

text_full = soup.get_text(separator=" ", strip=True)

# Collapse runs of whitespace - SEC HTML has tons of &nbsp; and tabs
text_full = re.sub(r'\s+', ' ', text_full).strip()

print(f"Raw HTML size:        {len(html_payload):>10,} chars")
print(f"Clean text size:      {len(text_full):>10,} chars")
print(f"Markup overhead:      {1 - len(text_full)/len(html_payload):.1%}")
print(f"\nFirst 500 chars of clean text:\n{text_full[:500]}")

# Locate every "Item" header in the text

In [ ]:
def show_item_occurrences(text: str, item_id: str, context_chars: int = 80) -> None:
    """Print every occurrence of 'Item X' with surrounding context."""
    pattern = re.compile(rf'\bItem\s+{re.escape(item_id)}\b', re.IGNORECASE)
    matches = list(pattern.finditer(text))
    
    print(f"=== 'Item {item_id}' — {len(matches)} occurrence(s) ===")
    for i, m in enumerate(matches, 1):
        snippet_start = max(0, m.start() - 5)
        snippet_end = min(len(text), m.end() + context_chars)
        snippet = text[snippet_start:snippet_end]
        print(f"  [{i}] pos={m.start():>7,}  '{snippet}'")
    print()


# Check the items we care about
show_item_occurrences(text_full, "1A")
show_item_occurrences(text_full, "1B")
show_item_occurrences(text_full, "7")
show_item_occurrences(text_full, "7A")
show_item_occurrences(text_full, "8")  # safety net for where Item 7A ends

# The extractor

In [ ]:
def find_section_body(text: str, item_id: str) -> int | None:
    """Find the start position of an Item's body section.
    
    Heuristic: match 'Item N.' (with required period+space) — the body
    header always uses this format, while cross-references use 
    'Item N of'/'Item N,' (no period). The TOC also uses the period 
    format but appears earlier, so taking the LAST match gives us the body.
    """
    pattern = re.compile(rf'\bItem\s+{re.escape(item_id)}\.\s', re.IGNORECASE)
    matches = list(pattern.finditer(text))
    return matches[-1].start() if matches else None


def extract_section(
    text: str, 
    item_id: str, 
    end_candidates: list[str],
) -> str | None:
    """Extract the body of an Item, ending at the earliest of `end_candidates`
    that appears after our start. Falls back to end-of-text if none found.
    
    Args:
        item_id: e.g. '1A', '7'
        end_candidates: items where this section could plausibly end, e.g.
            ['1B', '2'] for Item 1A — usually 1B, but Apple sometimes skips it.
    """
    start = find_section_body(text, item_id)
    if start is None:
        return None
    
    # Find the earliest candidate that comes after our start
    end = len(text)
    for next_id in end_candidates:
        next_start = find_section_body(text, next_id)
        if next_start is not None and next_start > start:
            end = min(end, next_start)
    
    return text[start:end].strip()


# Extract the two sections we care about
risk_factors = extract_section(text_full, "1A", end_candidates=["1B", "1C", "2"])
mda          = extract_section(text_full, "7",  end_candidates=["7A", "8"])

# Sanity check — these should be substantial chunks of text
print(f"Item 1A (Risk Factors): {len(risk_factors):>7,} chars" if risk_factors else "Item 1A: NOT FOUND")
print(f"Item 7  (MD&A):         {len(mda):>7,} chars"          if mda          else "Item 7:  NOT FOUND")

# Eyeball the extractions

In [ ]:
def preview(name: str, text: str | None, n: int = 400) -> None:
    if not text:
        print(f"=== {name}: NOT EXTRACTED ===\n")
        return
    print(f"=== {name} ({len(text):,} chars) ===")
    print(f"--- FIRST {n} chars ---")
    print(text[:n])
    print(f"\n--- LAST {n} chars ---")
    print(text[-n:])
    print()


preview("Item 1A — Risk Factors", risk_factors)
preview("Item 7  — MD&A",          mda)

# Save the extracted sections

In [ ]:
# PROCESSED_DIR was defined in Cell 7
def save_section(filename: str, content: str | None) -> None:
    if content is None:
        print(f"  (skipped {filename} — section not found)")
        return
    path = PROCESSED_DIR / filename
    path.write_text(content, encoding="utf-8")
    print(f"  ✓ {filename}: {path.stat().st_size / 1024:.1f} KB")


print(f"Saving to: {PROCESSED_DIR.relative_to(PROJECT_ROOT)}")
save_section("item_1a_risk_factors.txt", risk_factors)
save_section("item_7_mda.txt", mda)

# Re-extract AAPL via the module

In [ ]:
# Auto-reload the module if you edit it (handy during development)
%load_ext autoreload
%autoreload 2

from finintel.ingestion.parser import parse_filing, SECTION_SPECS

print("Configured filing types:", list(SECTION_SPECS.keys()))
print("\n10-K sections:")
for name, (item, ends) in SECTION_SPECS["10-K"].items():
    print(f"  {name:<14}  Item {item}  ends at {ends}")

In [ ]:
# Re-parse AAPL using the module — should match what we did inline
parsed = parse_filing(filing_path, filing_type="10-K")
print(f"Accession: {parsed.accession}")
for name, text in parsed.sections.items():
    print(f"  {name:<14}  {len(text):>7,} chars")

# Parse the full 10-K corpus

In [ ]:
from finintel.ingestion.parser import parse_filing

results = []  # list of (ticker, ParsedFiling)
errors = []   # list of (ticker, accession, error_message)

tickers = sorted(d.name for d in DATA_DIR.iterdir() if d.is_dir())
for ticker in tickers:
    form_dir = DATA_DIR / ticker / "10-K"
    if not form_dir.exists():
        continue
    for accession_dir in sorted(form_dir.iterdir()):
        sub_path = accession_dir / "full-submission.txt"
        if not sub_path.exists():
            errors.append((ticker, accession_dir.name, "no full-submission.txt"))
            continue
        try:
            parsed = parse_filing(sub_path, filing_type="10-K")
            if parsed is None:
                errors.append((ticker, accession_dir.name, "no primary 10-K document"))
            else:
                results.append((ticker, parsed))
        except Exception as e:
            errors.append((ticker, accession_dir.name, f"{type(e).__name__}: {e}"))

print(f"Parsed: {len(results)} filings")
print(f"Errors: {len(errors)}")
for t, a, e in errors:
    print(f"  - {t}/{a}: {e}")

# Inspect the size distribution

In [ ]:
print(f"{'Ticker':<8} {'Accession':<25} {'Risk Factors':>14} {'MD&A':>10}")
print("-" * 62)
for ticker, parsed in results:
    rf = len(parsed.sections.get("risk_factors", ""))
    mda = len(parsed.sections.get("mda", ""))
    flag = ""
    if rf < 5_000:
        flag += "  ⚠ tiny RF"
    if mda < 5_000:
        flag += "  ⚠ tiny MDA"
    if rf > 200_000:
        flag += "  ⚠ huge RF"
    print(f"{ticker:<8} {parsed.accession:<25} {rf:>14,} {mda:>10,}{flag}")

# Save all extracted text

In [ ]:
PROCESSED_BASE = PROJECT_ROOT / "data" / "processed"
saved = 0
for ticker, parsed in results:
    for section_name, content in parsed.sections.items():
        out_dir = PROCESSED_BASE / ticker / parsed.filing_type / parsed.accession
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / f"{section_name}.txt").write_text(content, encoding="utf-8")
        saved += 1

print(f"Saved {saved} section files under data/processed/")

# Diagnostic — what is in JPM's "MD&A"?

# Read the actual 395 chars

In [ ]:
jpm_results = [(t, p) for t, p in results if t == "JPM"]
ticker, parsed = jpm_results[0]
print(f"=== {ticker} {parsed.accession} — MD&A ({len(parsed.sections['mda'])} chars) ===\n")
print(parsed.sections["mda"])

# List document types in JPM's filing

In [ ]:
from collections import Counter

jpm_acc = parsed.accession
jpm_path = DATA_DIR / "JPM" / "10-K" / jpm_acc / "full-submission.txt"
raw = jpm_path.read_text(encoding="utf-8", errors="replace")

type_pattern = re.compile(r"<DOCUMENT>\s*<TYPE>([^\n<]+)")
types = [t.strip() for t in type_pattern.findall(raw)]

print(f"=== {jpm_acc} — {len(types)} documents ===\n")
for doc_type, count in Counter(types).most_common():
    print(f"  {count:>4d}  {doc_type}")

# Re-parse with the fix

In [ ]:
# autoreload should pick up the parser change automatically
results = []
errors = []
for ticker in tickers:
    form_dir = DATA_DIR / ticker / "10-K"
    if not form_dir.exists():
        continue
    for accession_dir in sorted(form_dir.iterdir()):
        sub_path = accession_dir / "full-submission.txt"
        if not sub_path.exists():
            continue
        try:
            parsed = parse_filing(sub_path, filing_type="10-K")
            if parsed:
                results.append((ticker, parsed))
        except Exception as e:
            errors.append((ticker, accession_dir.name, str(e)))

# Re-print the size table
print(f"{'Ticker':<8} {'Accession':<25} {'Risk Factors':>14} {'MD&A':>10}")
print("-" * 62)
for ticker, parsed in results:
    rf = len(parsed.sections.get("risk_factors", ""))
    mda = len(parsed.sections.get("mda", ""))
    flag = ""
    if rf < 5_000: flag += "  ⚠ tiny RF"
    if mda < 5_000: flag += "  ⚠ tiny MDA"
    print(f"{ticker:<8} {parsed.accession:<25} {rf:>14,} {mda:>10,}{flag}")

# Re-save the extracts

In [ ]:
import shutil

# Clear old extracts so nothing stale lingers
if (PROJECT_ROOT / "data" / "processed").exists():
    shutil.rmtree(PROJECT_ROOT / "data" / "processed")

saved = 0
for ticker, parsed in results:
    for section_name, content in parsed.sections.items():
        out_dir = PROCESSED_BASE / ticker / parsed.filing_type / parsed.accession
        out_dir.mkdir(parents=True, exist_ok=True)
        (out_dir / f"{section_name}.txt").write_text(content, encoding="utf-8")
        saved += 1

print(f"Saved {saved} section files")

# Verify JPM MD&A reads like real prose

In [ ]:
jpm_results_new = [(t, p) for t, p in results if t == "JPM"]
ticker, parsed = jpm_results_new[0]
mda = parsed.sections["mda"]
print(f"=== {ticker} {parsed.accession} — MD&A ({len(mda):,} chars) ===\n")
print("--- FIRST 600 chars ---")
print(mda[:600])
print("\n--- LAST 600 chars ---")
print(mda[-600:])